In [1]:
from matplotlib import rcParams, rc
rcParams.update({'figure.autolayout': True})
import torch
import matplotlib.pyplot as plt
#import matplotlib.animation as animation
import numpy as np
import math
from utils import Params
from tqdm import tqdm
#from IPython.display import HTML
#from IPython.display import Image
from GLOnet_thinfilm import GLOnet
import scipy.io as io
from TMM import *
from material_database import MatDatabase
from scipy.optimize import fsolve
import os
import random
import optuna

In [2]:
Vis = [0.45, 0.575]
params = Params()
params.N_layers = 7
params.pol = 'TM'
params.k =  2 * math.pi / torch.linspace(Vis[0], Vis[1], 100)
params.theta =   torch.tensor([0.]) 
params.n_top = torch.tensor([1.46]) #sustrato
params.n_bot = torch.tensor([1.]) #superestrato

params.user_define = False
## choose from material database
params.sensor = True
if params.sensor:
    params.materials_full_A =['TiO2_solgel_P45_agua', 'SiO2_solgel_P42_agua', 'ZrO2_solgel_P26_agua','TiO2_solgel_densa', 'SiO2_solgel_densa', 'ZrO2_solgel_densa']
    params.materials_full_B =['TiO2_solgel_P45_tolueno', 'SiO2_solgel_P42_tolueno', 'ZrO2_solgel_P26_tolueno','TiO2_solgel_densa', 'SiO2_solgel_densa', 'ZrO2_solgel_densa']
    params.materials_empty =['TiO2_solgel_P45_aire', 'SiO2_solgel_P42_aire', 'ZrO2_solgel_P26_aire','TiO2_solgel_densa', 'SiO2_solgel_densa','ZrO2_solgel_densa']
    #params.materials_full =['TiO2_solgel_P471_ll', 'SiO2_solgel_P363_ll', 'ZrO2_solgel_P296_ll']#,'TiO2_solgel_densa', 'SiO2_solgel_densa', 'ZrO2_solgel_densa']
    #params.materials_empty =['TiO2_solgel_P471_v', 'SiO2_solgel_P363_v', 'ZrO2_solgel_P296_v']#,'TiO2_solgel_densa', 'SiO2_solgel_densa','ZrO2_solgel_densa']
    params.matdatabase_full_A = MatDatabase(params.materials_full_A)
    params.matdatabase_full_B = MatDatabase(params.materials_full_B)
    params.matdatabase_empty = MatDatabase(params.materials_empty)
    params.n_database_full_A = params.matdatabase_full_A.interp_wv(2 * math.pi/params.k, params.materials_full_A, False) # number of materials x number of frequencies
    params.n_database_full_B = params.matdatabase_full_B.interp_wv(2 * math.pi/params.k, params.materials_full_B, False) # number of materials x number of frequencies
    params.n_database_empty = params.matdatabase_empty.interp_wv(2 * math.pi/params.k, params.materials_empty, False) # number of materials x number of frequencies
    params.M_materials = params.n_database_empty.size(0)
else:
    params.materials =['TiO2_solgel_P45_v', 'SiO2_solgel_P42_v', 'ZrO2_solgel_P26_v','TiO2_solgel_densa', 'SiO2_solgel_densa', 'ZrO2_solgel_densa']
    params.matdatabase = MatDatabase(params.materials)
    params.n_database = params.matdatabase.interp_wv(2 * math.pi/params.k, params.materials, False) # number of materials x number of frequencies
    params.M_materials = params.n_database.size(0)
    params.target_reflection = torch.zeros((1, params.k.size(0), 1, 1)) # 1 x number of frequencies x number of angles x (number of pol or 1)
    params.target_reflection[:,42:100, :, :] = 1

params.thickness_sup = 0.2 # [um]
params.thickness_l = 0.02 # [um]

params.net = 'Res'
params.res_layers = 16 
params.res_dim = 256 
params.noise_dim = 16 


In [ ]:
num_global_seeds = 2
base_results_root = 'optuna_shapley_runs' # Carpeta principal para todos los resultados
os.makedirs(base_results_root, exist_ok=True) # Crea la carpeta si no existe

for global_seed_idx in range(1, num_global_seeds + 1):
    current_fixed_seed = global_seed_idx # Usa la seed del bucle (1, 2, ..., 10)

    print(f"\n--- Iniciando estudio Optuna para SEED GLOBAL: {current_fixed_seed} ---")

    # Define la función objetivo para Optuna
    def objective(trial):
        # Crear una nueva instancia de params para este trial.
        # Esto es CRÍTICO para que cada trial tenga su propio conjunto de parámetros.
        trial_params = Params() 
        
        # --- Copia todos los parámetros BASE que no varían desde tu objeto 'params' global ---
        # (Este bloque es largo porque copias muchos parámetros fijos.
        #  Asegúrate de que 'params' (tu objeto Params inicial) tenga todos estos atributos).
        trial_params.N_layers = params.N_layers
        trial_params.pol = params.pol
        trial_params.k = params.k
        trial_params.theta = params.theta
        trial_params.n_top = params.n_top
        trial_params.n_bot = params.n_bot
        trial_params.user_define = params.user_define
        trial_params.sensor = params.sensor
        
        if trial_params.sensor:
            trial_params.materials_full_A = params.materials_full_A
            trial_params.materials_full_B = params.materials_full_B
            trial_params.materials_empty = params.materials_empty
            trial_params.matdatabase_full_A = params.matdatabase_full_A
            trial_params.matdatabase_full_B = params.matdatabase_full_B
            trial_params.matdatabase_empty = params.matdatabase_empty
            trial_params.n_database_full_A = params.n_database_full_A
            trial_params.n_database_full_B = params.n_database_full_B
            trial_params.n_database_empty = params.n_database_empty
        else:
            trial_params.materials = params.materials
            trial_params.matdatabase = params.matdatabase
            trial_params.n_database = params.n_database
            trial_params.target_reflection = params.target_reflection
        trial_params.M_materials = params.M_materials 

        trial_params.thickness_sup = params.thickness_sup
        trial_params.thickness_l = params.thickness_l
        trial_params.net = params.net
        trial_params.res_layers = params.res_layers
        trial_params.res_dim = params.res_dim
        trial_params.noise_dim = params.noise_dim
        # --- FIN de la copia de parámetros base ---

        # ---- Hiperparámetros que Optuna va a variar para cada TRIAL ----
        # Estos son los HPs que quieres optimizar/analizar con Shapley
        trial_params.lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
        trial_params.beta1 = trial.suggest_float('beta1', 0.8, 0.99)
        trial_params.beta2 = trial.suggest_float('beta2', 0.9, 0.999)
        trial_params.weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
        trial_params.step_size = trial.suggest_categorical('step_size', [250, 500, 750, 1000])
        trial_params.gamma = trial.suggest_float('gamma', 0.1, 0.9, step=0.1)
        trial_params.numIter = trial.suggest_int('numIter', 300, 700, step=100)
        trial_params.alpha_sup = trial.suggest_float('alpha_sup', 2.0, 4.0, step=0.5)
        trial_params.batch_size = trial.suggest_categorical('batch_size', [128, 256, 300, 512])
        trial_params.sigma = trial.suggest_float('sigma', 0.1, 0.8, step=0.1)
        
        # --- La SEED global FIJA para este estudio Optuna ---
        trial_params.seed = current_fixed_seed 
        
        # Establecer las seeds para esta prueba específica (mismo para todos los trials de este estudio)
        torch.manual_seed(trial_params.seed)
        random.seed(trial_params.seed)
        np.random.seed(trial_params.seed)

        # Construir la ruta base para este trial (ej. optuna_shapley_runs/global_seed_1/trial_0)
        trial_base_dir = os.path.join(base_results_root, 
                                        f"global_seed_{current_fixed_seed}",
                                        f"trial_{trial.number}")
        
        # La ruta FINAL que GLOnet usará. ¡AHORA INCLUYE LA CARPETA '/seed_X/'!
        trial_params.ruta = os.path.join(trial_base_dir, f"seed_{trial_params.seed}") 

        # Asegúrate de que este directorio final (incluyendo /seed_X/) exista
        os.makedirs(trial_params.ruta, exist_ok=True)
        seed_specific_dir = os.path.join(trial_params.ruta, f"seed_{trial_params.seed}")
        os.makedirs(seed_specific_dir, exist_ok=True)

        # Entrenar el modelo con los parámetros del trial actual
        glonet = GLOnet(trial_params)
        final_loss = glonet.train() # glonet.train() DEBE retornar la pérdida final
        glonet.viz_training() 
        return final_loss # Optuna intentará minimizar este valor

    # --- Configurar y ejecutar el estudio de Optuna para esta SEED GLOBAL ---
    study_name = f"glonet_hp_study_seed_{current_fixed_seed}"

    # Directorio donde se guardarán los archivos .db de Optuna para esta seed global
    db_dir = os.path.join(base_results_root, f"global_seed_{current_fixed_seed}")
    os.makedirs(db_dir, exist_ok=True) # CREA ESTE DIRECTORIO SI NO EXISTE

    storage_path = os.path.join(db_dir, f"{study_name}.db") 
    storage_name = f"sqlite:///{storage_path}" # Guarda el historial en una base de datos

    study = optuna.create_study(
        direction="minimize", # Queremos minimizar la pérdida
        study_name=study_name,
        storage=storage_name,
        load_if_exists=True # Carga un estudio existente si ya lo has corrido antes
    )

    n_trials = 50 # Número de combinaciones de hiperparámetros a probar por cada seed global
    study.optimize(objective, n_trials=n_trials, n_jobs=1) # n_jobs=1 para ejecutar secuencialmente.

    print(f"\n--- Estudio Optuna para SEED GLOBAL {current_fixed_seed} Terminado ---")
    print(f"Mejor trial (pérdida mínima): {study.best_trial.value}")
    print(f"Mejores hiperparámetros: {study.best_trial.params}")

    # --- Exportar los resultados de este estudio para el análisis Shapley ---
    df_results_for_shapley = study.trials_dataframe()

    # Seleccionar solo las columnas de hiperparámetros y la métrica de rendimiento
    hp_columns = [col for col in df_results_for_shapley.columns if col.startswith('params_')]
    df_shap_analysis = df_results_for_shapley[hp_columns + ['value']]

    # Renombrar las columnas para mayor claridad (quitar 'params_')
    df_shap_analysis.columns = [col.replace('params_', '') for col in hp_columns] + ['final_loss']

    # Guardar el DataFrame para usarlo en tu script de análisis SHAP posterior
    output_csv_path = os.path.join(base_results_root, 
                                   f"global_seed_{current_fixed_seed}",
                                   f'optuna_results_seed_{current_fixed_seed}.csv')
    df_shap_analysis.to_csv(output_csv_path, index=False)
    print(f"Resultados para análisis Shapley guardados en: {output_csv_path}")